# 09.07 - Small CNN Coding

**Notebook type:** Solution notebook with full working code and test cases.

**Daily output:** a small CNN baseline trained on a synthetic image classification dataset, with train and validation loss recorded.

Today turns Day 08 shape math into a real model: conv blocks, batch norm, activation, pooling, classifier head, `CrossEntropyLoss`, and a complete train/eval loop.


## Core Ideas

A compact CNN classifier usually has:

- a feature extractor: repeated convolution blocks
- nonlinear activations: usually ReLU or GELU
- normalization: often BatchNorm for small CNNs
- pooling or stride: shrinks spatial size
- classifier head: flatten or global average pool, then linear layers

For multi-class classification with class IDs, use `nn.CrossEntropyLoss`. The model should output raw logits shaped `[batch, num_classes]`; do not apply softmax before the loss.


In [ ]:
import random
import numpy as np

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    torch = None
    nn = None
    DataLoader = None
    TensorDataset = None
    TORCH_AVAILABLE = False
    print("PyTorch is not installed. Complete this notebook in an environment with torch.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)
else:
    device = None


## Prepared Image Data

Run this cell before the exercises. The synthetic image data is provided so the learning work starts at dataset inspection, batching, modeling, and training.


In [ ]:
def make_synthetic_image_dataset(n_per_class=80, image_size=16, noise=0.12, seed=42):
    if not TORCH_AVAILABLE:
        raise ImportError("PyTorch is required for this exercise.")

    generator = torch.Generator().manual_seed(seed)
    images = []
    labels = []

    for class_id in range(3):
        for _ in range(n_per_class):
            img = torch.zeros(1, image_size, image_size, dtype=torch.float32)
            if class_id == 0:
                img[:, :, image_size // 2 - 1:image_size // 2 + 1] = 1.0
            elif class_id == 1:
                img[:, image_size // 2 - 1:image_size // 2 + 1, :] = 1.0
            else:
                for i in range(image_size):
                    img[:, i, i] = 1.0
                    if i + 1 < image_size:
                        img[:, i, i + 1] = 1.0

            img = img + noise * torch.randn(img.shape, generator=generator)
            img = img.clamp(0.0, 1.0)
            images.append(img)
            labels.append(class_id)

    X = torch.stack(images)
    y = torch.tensor(labels, dtype=torch.long)
    perm = torch.randperm(len(y), generator=generator)
    return X[perm], y[perm]

if TORCH_AVAILABLE:
    X, y = make_synthetic_image_dataset()
    print("X:", X.shape, X.dtype)
    print("y:", y.shape, y.dtype, y.unique().tolist())


## Exercise 09-A: Dataset Sanity Checks

Use the prepared tensors `X` and `y`. Write a small summary helper that checks image shape, label shape, dtypes, and label values before the data reaches a model.


In [ ]:
def summarize_dataset(X, y):
    return {
        "image_shape": tuple(X.shape),
        "label_shape": tuple(y.shape),
        "image_dtype": X.dtype,
        "label_dtype": y.dtype,
        "label_values": sorted(y.unique().tolist()),
    }


if TORCH_AVAILABLE:
    summary = summarize_dataset(X, y)
    print(summary)


## Exercise 09-B: DataLoaders

Split the synthetic data into train and validation sets. Use shuffled training batches and deterministic validation batches.


In [ ]:
def build_loaders(X, y, batch_size=32, train_frac=0.8, seed=42):
    if not TORCH_AVAILABLE:
        raise ImportError("PyTorch is required for this exercise.")

    n = len(y)
    generator = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=generator)
    train_n = int(train_frac * n)
    train_idx = perm[:train_n]
    val_idx = perm[train_n:]

    train_ds = TensorDataset(X[train_idx], y[train_idx])
    val_ds = TensorDataset(X[val_idx], y[val_idx])
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

if TORCH_AVAILABLE:
    train_loader, val_loader = build_loaders(X, y)
    xb, yb = next(iter(train_loader))
    print("batch:", xb.shape, yb.shape, yb.dtype)


## Exercise 09-C: CNN Building Blocks

Implement a reusable conv block:

`Conv2d -> BatchNorm2d -> ReLU`

Then build a small CNN with two pooling stages and a classifier head.


In [ ]:
class ConvBlock(nn.Module if TORCH_AVAILABLE else object):
    def __init__(self, in_channels, out_channels):
        if not TORCH_AVAILABLE:
            return
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class SmallCNN(nn.Module if TORCH_AVAILABLE else object):
    def __init__(self, num_classes=3):
        if not TORCH_AVAILABLE:
            return
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 16),
            nn.MaxPool2d(2),
            ConvBlock(16, 32),
            nn.MaxPool2d(2),
            ConvBlock(32, 64),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

if TORCH_AVAILABLE:
    model = SmallCNN(num_classes=3).to(device)
    with torch.no_grad():
        logits = model(xb.to(device))
    print(model)
    print("logits:", logits.shape)


## Exercise 09-D: Train and Evaluate

Write one training epoch and one evaluation function. Track average loss and accuracy.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_seen += len(yb)

    return total_loss / total_seen, total_correct / total_seen

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * len(yb)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_seen += len(yb)

    return {"loss": total_loss / total_seen, "accuracy": total_correct / total_seen}


## Exercise 09-E: Record Curves

Train the CNN for a small number of epochs and record `train_loss`, `train_acc`, `val_loss`, and `val_acc`.


In [ ]:
history = []

if TORCH_AVAILABLE:
    model = SmallCNN(num_classes=3).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(1, 6):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate(model, val_loader, criterion, device)
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
        }
        history.append(row)
        print(row)
else:
    print("Skipped training because torch is not installed.")


## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 09 tests passed`.


In [ ]:
def run_day09_tests():
    if not TORCH_AVAILABLE:
        print("Day 09 tests skipped because PyTorch is not installed.")
        return

    required_names = [
        "make_synthetic_image_dataset",
        "summarize_dataset",
        "build_loaders",
        "ConvBlock",
        "SmallCNN",
        "train_one_epoch",
        "evaluate",
    ]
    for name in required_names:
        assert name in globals(), f"Missing function or class: {name}"
        assert callable(globals()[name]), f"{name} must be callable"

    X_test, y_test = make_synthetic_image_dataset(n_per_class=6, image_size=16, seed=123)
    assert X_test.shape == (18, 1, 16, 16), f"Unexpected X shape: {X_test.shape}"
    assert y_test.shape == (18,), f"Unexpected y shape: {y_test.shape}"
    assert X_test.dtype == torch.float32
    assert y_test.dtype == torch.long
    assert set(y_test.tolist()) == {0, 1, 2}

    summary = summarize_dataset(X_test, y_test)
    assert summary["image_shape"] == (18, 1, 16, 16)
    assert summary["label_shape"] == (18,)
    assert summary["image_dtype"] == torch.float32
    assert summary["label_dtype"] == torch.long
    assert summary["label_values"] == [0, 1, 2]

    train_loader_test, val_loader_test = build_loaders(X_test, y_test, batch_size=6, train_frac=0.67, seed=123)
    xb, yb = next(iter(train_loader_test))
    assert xb.ndim == 4 and xb.shape[1:] == (1, 16, 16)
    assert yb.dtype == torch.long

    model_test = SmallCNN(num_classes=3).to(device)
    with torch.no_grad():
        logits = model_test(xb.to(device))
    assert logits.shape == (xb.shape[0], 3), f"Unexpected logits shape: {logits.shape}"

    criterion_test = nn.CrossEntropyLoss()
    optimizer_test = torch.optim.Adam(model_test.parameters(), lr=0.01)
    train_loss, train_acc = train_one_epoch(model_test, train_loader_test, criterion_test, optimizer_test, device)
    assert isinstance(train_loss, float)
    assert 0.0 <= train_acc <= 1.0

    metrics = evaluate(model_test, val_loader_test, criterion_test, device)
    assert {"loss", "accuracy"}.issubset(metrics.keys())
    assert isinstance(metrics["loss"], float)
    assert 0.0 <= metrics["accuracy"] <= 1.0

    print("Day 09 tests passed")

run_day09_tests()


## Day 09 Checklist

Before trusting a CNN baseline, verify batch shape, logits shape, label dtype, loss decreases, validation is run with `eval()` and `no_grad()`, and the recorded metrics are validation metrics rather than training-only numbers.
